In [ ]:
# ================== MOUNT DRIVE ==================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import random
import csv
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, roc_curve, auc, accuracy_score, recall_score, classification_report
from sklearn.preprocessing import label_binarize
from tqdm import tqdm
import timm
import seaborn as sns
import cv2

# ========================== CONFIG CẬP NHẬT (STRATEGY SOTA) ==========================
SEEDS = [42, 52, 62]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device đang sử dụng:", device)

BASE_DATA_DIR = "/content/drive/MyDrive/VĐHĐ_TTNT/Skin cancer ISIC The International Skin Imaging Collaboration"
TRAIN_DIR = os.path.join(BASE_DATA_DIR, "Train")
TEST_DIR = os.path.join(BASE_DATA_DIR, "Test")

# Đổi tên thư mục lưu để lưu kết quả của chiến thuật mới
SAVE_ROOT = "/content/drive/MyDrive/VĐHĐ_TTNT/VGG16"
os.makedirs(SAVE_ROOT, exist_ok=True)

# --- Thông số mô hình ---
NUM_CLASSES = 9
BATCH_SIZE = 16
NUM_EPOCHS = 70


USE_HAIR_REMOVAL = False     # Tắt xóa lông để giữ texture gốc
GRAD_CLIP = 1.0              # Gradient Clipping để ổn định training
# Bảng trọng số nhẹ nhàng hơn để tránh mô hình bị "ép" quá mức
# Thứ tự: [actinic, basal, dermato, melanoma, nevus, pigmented, seborrheic, squamous, vascular]
MANUAL_WEIGHTS = [2.5, 1.0, 1.2, 5.0, 0.5, 1.0, 3.0, 1.2, 1.0]

# --- Hyperparameters ---
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.05
DROPOUT = 0.5 #0.5 cho VGG16, 0.3 cho EfficientNetB0-B2-B3
LABEL_SMOOTHING = 0.1
FOCAL_GAMMA = 2.0

# --- Xử lý dữ liệu ---
T_MAX = 50
PATIENCE = 15
MIXUP_ALPHA = 0.2 # Giữ 0.2 để phối hợp tốt với WeightedSampler
IMAGE_SIZE = 224
label_map = {
    0: 'actinic keratosis',
    1: 'basal cell carcinoma',
    2: 'dermatofibroma',
    3: 'melanoma',
    4: 'nevus',
    5: 'pigmented benign keratosis',
    6: 'seborrheic keratosis',
    7: 'squamous cell carcinoma',
    8: 'vascular lesion'
}
# ========================== HELPER FUNCTIONS ==========================
def hair_removal(image_pil):
    img = np.array(image_pil)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, thresh = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    dst = cv2.inpaint(img, thresh, 1, cv2.INPAINT_TELEA)
    return Image.fromarray(dst)

def mixup_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

# ========================== DATA COMPONENTS ==========================
# BẮT BUỘC đổi về 224x224 — đây là thay đổi quan trọng nhất
class RandomAugmentationPerImage:
    def __init__(self):
        self.resize = transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0))
        self.augs = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(30),
            transforms.ColorJitter(brightness=0.1, contrast=0.1,
                                   saturation=0.1, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def __call__(self, img):
        return self.augs(self.resize(img))

class CustomDataset(Dataset):
    def __init__(self, paths, labels, transform=None, use_hair_removal=False):
        self.paths = np.array(paths)
        self.labels = np.array(labels)
        self.transform = transform
        self.use_hair_removal = use_hair_removal

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
            # Ưu tiên 4: Kiểm soát Xóa lông qua biến cấu hình
            if self.use_hair_removal:
                img = hair_removal(img)

            label = int(self.labels[idx])
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception as e:
            # Trả về ảnh trống nếu file lỗi để tránh dừng ngang quá trình train
            return torch.zeros((3, IMAGE_SIZE, IMAGE_SIZE)), int(self.labels[idx])

# ========================== LOSS & EARLY STOPPING ==========================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        # Tích hợp Label Smoothing trực tiếp vào Focal Loss
        log_probs = torch.log_softmax(inputs, dim=-1)
        targets_smooth = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_smooth = targets_smooth * (1 - self.smoothing) + self.smoothing / NUM_CLASSES

        ce_loss = (-targets_smooth * log_probs).sum(dim=-1)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss)

        if self.weight is not None:
            # Áp dụng trọng số từ MANUAL_WEIGHTS
            w = self.weight[targets]
            focal_loss = focal_loss * w

        return focal_loss.mean()

class EarlyStopping:
    def __init__(self, patience=10):
        self.patience = patience
        self.best = None
        self.counter = 0
        self.stop = False
    def __call__(self, metric):
        if self.best is None or metric > self.best:
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

# === KHỞI TẠO BIẾN ===
transform_train = RandomAugmentationPerImage()
transform_val = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),   # 256 → 224
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
train_dataset_raw = datasets.ImageFolder(TRAIN_DIR)
test_dataset = datasets.ImageFolder(TEST_DIR)
# ========================== MULTI SEED (SOTA STRATEGY) ==========================
for SEED in SEEDS:
    print(f"\n========== BẮT ĐẦU SEED {SEED} ==========")
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    SEED_DIR = os.path.join(SAVE_ROOT, f"seed_{SEED}")
    os.makedirs(SEED_DIR, exist_ok=True)

    # 1. Khởi tạo Logging
    csv_file = open(os.path.join(SEED_DIR, "training_log.csv"), "w", newline="")
    csv_writer = csv.writer(csv_file)
    header = ["epoch", "lr", "train_loss", "train_acc", "val_loss", "val_acc", "val_f1", "val_recall"]
    for i in range(NUM_CLASSES): header.append(f"{label_map[i]}_F1")
    csv_writer.writerow(header)
    txt_log = open(os.path.join(SEED_DIR, "training_log.txt"), "w")

    # 2. Chia dữ liệu GỐC (Sạch, không nhân bản thủ công)
    raw_paths = np.array([train_dataset_raw.imgs[i][0] for i in range(len(train_dataset_raw.imgs))])
    raw_labels = np.array([train_dataset_raw.imgs[i][1] for i in range(len(train_dataset_raw.imgs))])

    tr_p, va_p, tr_l, va_l = train_test_split(
        raw_paths, raw_labels, test_size=0.2, stratify=raw_labels, random_state=SEED
    )

    # 3. ƯU TIÊN 2: WeightedRandomSampler (Cân bằng động)
    class_counts = Counter(tr_l)
    # Trọng số lấy mẫu = 1 / số lượng mẫu của lớp đó
    weights_per_class = {cls: 1.0 / count for cls, count in class_counts.items()}
    sample_weights = [weights_per_class[l] for l in tr_l]
    # Tạo Sampler: replacement=True để cho phép lấy lặp lại các lớp hiếm
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(tr_l), replacement=True)

    # 4. Khởi tạo Loaders (Dùng USE_HAIR_REMOVAL từ Config)
    train_loader = DataLoader(
        CustomDataset(tr_p, tr_l, transform_train, use_hair_removal=USE_HAIR_REMOVAL),
        batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        CustomDataset(va_p, va_l, transform_val, use_hair_removal=USE_HAIR_REMOVAL),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    test_paths = [p for p, _ in test_dataset.imgs]
    test_labels = [l for _, l in test_dataset.imgs]
    test_loader = DataLoader(
          CustomDataset(test_paths, test_labels, transform_val, use_hair_removal=USE_HAIR_REMOVAL),
          batch_size=BATCH_SIZE, shuffle=False, num_workers=2
      )
    # 5. Khởi tạo Model & Optimizer
    model = timm.create_model('vgg16',pretrained=True, num_classes=NUM_CLASSES,drop_rate=DROPOUT).to(device)
    #Trọng số Focal Loss nhẹ hơn (giảm thiên kiến cực đoan)
    w = torch.tensor(MANUAL_WEIGHTS, dtype=torch.float32).to(device)
    criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=w, smoothing=LABEL_SMOOTHING)

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LEARNING_RATE, steps_per_epoch=len(train_loader), epochs=NUM_EPOCHS
    )
    early = EarlyStopping(patience=PATIENCE)

    #AMP Scaler
    scaler = torch.amp.GradScaler('cuda')

    history = {'t_loss': [], 't_acc': [], 'v_loss': [], 'v_acc': [], 'v_f1': [], 'v_rec': []}
    best_f1, best_path = 0, os.path.join(SEED_DIR, "best_model.pt")

    # ========================== VÒNG LẶP HUẤN LUYỆN ==========================
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        pbar = tqdm(train_loader, desc=f"Seed {SEED} Ep {epoch}")

        for imgs, lbl in pbar:
            imgs, lbl = imgs.to(device), lbl.to(device)
            optimizer.zero_grad()

            # ƯU TIÊN 1: AMP Autocast (FP16)
            with torch.amp.autocast('cuda'):
                imgs_m, la, lb, lam = mixup_data(imgs, lbl, MIXUP_ALPHA)
                out = model(imgs_m)
                loss = lam * criterion(out, la) + (1 - lam) * criterion(out, lb)

            scaler.scale(loss).backward()

            # ƯU TIÊN 1: Gradient Clipping (Chống nổ gradient)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            train_loss += loss.item()
            train_total += lbl.size(0)
            # Tính Acc chuẩn cho Mixup
            train_correct += (lam * (out.argmax(1) == la).float() + (1 - lam) * (out.argmax(1) == lb).float()).sum().item()
            pbar.set_postfix(acc=train_correct/train_total)

        # VALIDATION
        model.eval()
        v_preds, v_labels, v_loss_sum = [], [], 0
        with torch.no_grad():
            for imgs, lbl in val_loader:
                imgs, lbl = imgs.to(device), lbl.to(device)
                with torch.amp.autocast('cuda'):
                    out = model(imgs)
                    v_loss_sum += criterion(out, lbl).item()
                v_preds.extend(out.argmax(1).cpu().numpy()); v_labels.extend(lbl.cpu().numpy())

        # Tính toán Metrics
        t_loss_avg = train_loss / len(train_loader)
        t_acc_avg = train_correct / train_total
        v_loss_avg = v_loss_sum / len(val_loader)
        v_acc_avg = accuracy_score(v_labels, v_preds)
        v_f1 = f1_score(v_labels, v_preds, average='macro')
        v_rec = recall_score(v_labels, v_preds, average='macro')
        class_f1 = f1_score(v_labels, v_preds, average=None)

        # Lưu history
        history['t_loss'].append(t_loss_avg); history['t_acc'].append(t_acc_avg)
        history['v_loss'].append(v_loss_avg); history['v_acc'].append(v_acc_avg)
        history['v_f1'].append(v_f1); history['v_rec'].append(v_rec)

        # Print & Log
        lr_now = optimizer.param_groups[0]['lr']
        summary = f"Epoch {epoch}/{NUM_EPOCHS} | LR: {lr_now:.6f} | Train Acc: {t_acc_avg:.4f} | Val F1: {v_f1:.4f} | Val Acc: {v_acc_avg:.4f}"
        print(summary); txt_log.write(summary + "\n")

        row = [epoch, lr_now, t_loss_avg, t_acc_avg, v_loss_avg, v_acc_avg, v_f1, v_rec]
        for i in range(NUM_CLASSES): row.append(class_f1[i])
        csv_writer.writerow(row); csv_file.flush()

        if v_f1 > best_f1:
            best_f1, best_epoch = v_f1, epoch
            torch.save(model.state_dict(), best_path)
            print(f"  --> Đã lưu Best Model (F1: {v_f1:.4f})")

        early(v_f1)
        if early.stop: break

    # 6. VẼ BIỂU ĐỒ (Sau mỗi Seed)
    plt.figure(figsize=(18, 5))
    plt.subplot(1,3,1); plt.plot(history['t_loss'], label='Train'); plt.plot(history['v_loss'], label='Val'); plt.title('Loss'); plt.legend()
    plt.subplot(1,3,2); plt.plot(history['t_acc'], label='Train'); plt.plot(history['v_acc'], label='Val'); plt.title('Accuracy'); plt.legend()
    plt.subplot(1,3,3); plt.plot(history['v_f1'], label='F1'); plt.plot(history['v_rec'], label='Recall'); plt.title('Macro F1 & Recall'); plt.legend()
    plt.savefig(os.path.join(SEED_DIR, "curves.png")); plt.show()

    csv_file.close(); txt_log.close()
    # ========================== TEST PHASE (SOTA VERSION) ==========================
    print(f"\n--- BẮT ĐẦU KIỂM THỬ CUỐI CÙNG SEED {SEED} ---")
    model.load_state_dict(torch.load(best_path))
    model.eval()

    preds_all, labels_all, probs_all = [], [], []

    # test_loader đã tự động áp dụng cấu hình USE_HAIR_REMOVAL từ Dataset
    with torch.no_grad():
        for imgs, lbl in tqdm(test_loader, desc=f"Final Test Seed {SEED}"):
            imgs = imgs.to(device)

            # ƯU TIÊN 1: AMP Autocast cho Inference (Tăng tốc TTA)
            with torch.amp.autocast('cuda'):
                # --- Áp dụng TTA (Test-Time Augmentation) 4 góc ---
                out1 = model(imgs)                               # Gốc
                out2 = model(torch.flip(imgs, [3]))              # Lật ngang
                out3 = model(torch.flip(imgs, [2]))              # Lật dọc
                out4 = model(torch.rot90(imgs, 1, [2, 3]))       # Xoay 90 độ

                # Tính trung bình cộng xác suất (Dùng Softmax trước khi cộng)
                p1 = torch.softmax(out1, dim=1)
                p2 = torch.softmax(out2, dim=1)
                p3 = torch.softmax(out3, dim=1)
                p4 = torch.softmax(out4, dim=1)

                mean_probs = (p1 + p2 + p3 + p4) / 4
                preds = mean_probs.argmax(dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(lbl.numpy())
            probs_all.extend(mean_probs.cpu().numpy())

    labels_all = np.array(labels_all)
    preds_all = np.array(preds_all)
    probs_all = np.array(probs_all)

    # 1. Vẽ Confusion Matrix (Chuẩn hóa hiển thị)
    class_names = [label_map[i] for i in range(NUM_CLASSES)]
    cm = confusion_matrix(labels_all, preds_all)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Seed {SEED} (TTA Enabled)")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(SEED_DIR, "confusion_matrix.png"))
    plt.show()
    plt.close() # Giải phóng RAM

    # 2. Vẽ ROC Curve & Tính AUC chi tiết
    y_bin = label_binarize(labels_all, classes=list(range(NUM_CLASSES)))
    plt.figure(figsize=(10, 8))
    for i in range(NUM_CLASSES):
        if np.sum(y_bin[:, i]) > 0:
            fpr, tpr, _ = roc_curve(y_bin[:, i], probs_all[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f'{label_map[i]} (AUC = {roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.title(f'ROC Curves - Seed {SEED}')
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(alpha=0.3)
    plt.savefig(os.path.join(SEED_DIR, "roc_curve.png"))
    plt.show()
    plt.close()

    # 3. Báo cáo phân loại và Lưu trữ
    report = classification_report(labels_all, preds_all, target_names=class_names)
    print(f"\n===== RESULT SUMMARY SEED {SEED} =====")
    print(report)

    with open(os.path.join(SEED_DIR, "final_report.txt"), "w", encoding="utf-8") as f:
        f.write(f"Best Epoch: {best_epoch}\n")
        f.write(report)

    # DỌN DẸP BỘ NHỚ TRƯỚC KHI SANG SEED TIẾP THEO
    del model
    torch.cuda.empty_cache()

print("\n--- TOÀN BỘ QUÁ TRÌNH HOÀN TẤT ---")